# RTS-GMLC Grid Visualization

Interactive map of the RTS-GMLC network built from `bus.csv`, `gen.csv`, and `branch.csv`
in `src/Prescient_initialization/synthetic_gmlc/RTS_Data/SourceData`.

**Interactions**
- **Click a bus** -> table of its generators with `PMax MW`, `Ramp Rate MW/Min`, `Min Down Time Hr`, `Min Up Time Hr`.
- **Click a transmission line** (its square midpoint marker) -> `From Bus` / `To Bus` and `LTE Rating` / `STE Rating`.

Requires `plotly` and `ipywidgets` (`pip install plotly ipywidgets`). Click callbacks need a widget-enabled
frontend; if they do not fire, the hover tooltips carry the same information (see the static fallback below).

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

# Resolve the SourceData directory relative to this notebook (src/notebook/).
NB_DIR = Path.cwd()
CANDIDATES = [
    NB_DIR / ".." / "Prescient_initialization" / "synthetic_gmlc" / "RTS_Data" / "SourceData",
    NB_DIR / "src" / "Prescient_initialization" / "synthetic_gmlc" / "RTS_Data" / "SourceData",
]
DATA_DIR = next((p.resolve() for p in CANDIDATES if p.exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError("Could not locate RTS_Data/SourceData; set DATA_DIR manually.")
print("DATA_DIR:", DATA_DIR)

DATA_DIR: C:\Users\JKLCh\Desktop\ND_PhD\Research\GNN_PCM\src\Prescient_initialization\synthetic_gmlc\RTS_Data\SourceData


In [3]:
bus = pd.read_csv(DATA_DIR / "bus.csv")
gen = pd.read_csv(DATA_DIR / "gen.csv")
branch = pd.read_csv(DATA_DIR / "branch.csv")

bus["Bus ID"] = bus["Bus ID"].astype(int)
gen["Bus ID"] = gen["Bus ID"].astype(int)
branch["From Bus"] = branch["From Bus"].astype(int)
branch["To Bus"] = branch["To Bus"].astype(int)

print(f"{len(bus)} buses, {len(gen)} generators, {len(branch)} branches")
bus.head()

73 buses, 158 generators, 120 branches


,Bus ID,Bus Name,BaseKV,Bus Type,MW Load,MVAR Load,V Mag,V Angle,MW Shunt G,MVAR Shunt B,Area,Sub Area,Zone,lat,lng
0,101,Abel,138.0,PV,108.0,22.0,1.04777,-7.74152,0.0,0.0,1,11.0,11.0,33.396103,-113.835642
1,102,Adams,138.0,PV,97.0,20.0,1.04783,-7.81784,0.0,0.0,1,11.0,12.0,33.357678,-113.825933
2,103,Adler,138.0,PQ,180.0,37.0,1.01085,-7.21090,0.0,0.0,1,11.0,11.0,33.536833,-114.670399
3,104,Agricola,138.0,PQ,74.0,15.0,1.01765,-10.56614,0.0,0.0,1,11.0,11.0,33.812304,-113.825419
4,105,Aiken,138.0,PQ,71.0,14.0,1.03568,-10.70887,0.0,0.0,1,11.0,11.0,33.659560,-113.999023


In [4]:
# Node coordinates come straight from the bus lat/lng columns.
pos = {int(r["Bus ID"]): (float(r["lng"]), float(r["lat"])) for _, r in bus.iterrows()}

GEN_COLS = ["GEN UID", "Unit Type", "Fuel", "PMax MW", "PMin MW",
            "Ramp Rate MW/Min", "Min Down Time Hr", "Min Up Time Hr"]

gen_by_bus = {b: g[GEN_COLS].reset_index(drop=True) for b, g in gen.groupby("Bus ID")}

summary = (gen.groupby("Bus ID")
              .agg(n_gen=("GEN UID", "size"), pmax_total=("PMax MW", "sum"))
              .reindex(bus["Bus ID"]).fillna(0))
bus["n_gen"] = summary["n_gen"].values
bus["pmax_total"] = summary["pmax_total"].values
bus.head()

,Bus ID,Bus Name,BaseKV,Bus Type,MW Load,MVAR Load,V Mag,V Angle,MW Shunt G,MVAR Shunt B,Area,Sub Area,Zone,lat,lng,n_gen,pmax_total
0,101,Abel,138.0,PV,108.0,22.0,1.04777,-7.74152,0.0,0.0,1,11.0,11.0,33.396103,-113.835642,8.0,296.6
1,102,Adams,138.0,PV,97.0,20.0,1.04783,-7.81784,0.0,0.0,1,11.0,12.0,33.357678,-113.825933,6.0,242.9
2,103,Adler,138.0,PQ,180.0,37.0,1.01085,-7.21090,0.0,0.0,1,11.0,11.0,33.536833,-114.670399,1.0,61.5
3,104,Agricola,138.0,PQ,74.0,15.0,1.01765,-10.56614,0.0,0.0,1,11.0,11.0,33.812304,-113.825419,1.0,26.8
4,105,Aiken,138.0,PQ,71.0,14.0,1.03568,-10.70887,0.0,0.0,1,11.0,11.0,33.659560,-113.999023,0.0,0.0


In [5]:
# ---- Line traces: the drawn segments are inert; an invisible-ish midpoint marker
# ---- per branch carries the hover text and receives the click.
edge_x, edge_y = [], []
mid_x, mid_y, mid_text = [], [], []
branch_rows = []

for _, r in branch.iterrows():
    f, t = int(r["From Bus"]), int(r["To Bus"])
    if f not in pos or t not in pos:
        continue
    x0, y0 = pos[f]
    x1, y1 = pos[t]
    edge_x += [x0, x1, None]
    edge_y += [y0, y1, None]
    mid_x.append((x0 + x1) / 2)
    mid_y.append((y0 + y1) / 2)
    mid_text.append(
        "<b>Line " + str(r["UID"]) + "</b><br>"
        + f"From Bus: {f}<br>To Bus: {t}<br>"
        + f"LTE Rating: {r['LTE Rating']} MVA<br>STE Rating: {r['STE Rating']} MVA"
    )
    branch_rows.append(r)

branch_plot = pd.DataFrame(branch_rows).reset_index(drop=True)

edge_trace = go.Scatter(x=edge_x, y=edge_y, mode="lines",
                        line=dict(color="rgba(120,120,140,0.75)", width=1.4),
                        hoverinfo="skip", showlegend=False, name="Lines")

line_trace = go.Scatter(x=mid_x, y=mid_y, mode="markers",
                        marker=dict(size=9, symbol="square",
                                    color="rgba(120,120,140,0.35)"),
                        text=mid_text, hoverinfo="text",
                        customdata=np.arange(len(branch_plot)),
                        showlegend=False, name="Line (click)")

In [7]:
# ---- Bus trace: marker size ~ installed capacity, color ~ area.
bus_text = []
for _, r in bus.iterrows():
    bus_text.append(
        f"<b>Bus {int(r['Bus ID'])} - {r['Bus Name']}</b><br>"
        f"{r['BaseKV']} kV, type {r['Bus Type']}<br>"
        f"Area {r['Area']} / Zone {r['Zone']}<br>"
        f"Load: {r['MW Load']} MW<br>"
        f"Generators: {int(r['n_gen'])} (PMax total {r['pmax_total']:.0f} MW)"
        "<br><i>click for generator detail</i>"
    )

pmax_max = max(float(bus["pmax_total"].max()), 1.0)
sizes = 8 + 22 * np.sqrt(bus["pmax_total"].values / pmax_max)

bus_trace = go.Scatter(
    x=[pos[b][0] for b in bus["Bus ID"]],
    y=[pos[b][1] for b in bus["Bus ID"]],
    mode="markers",
    marker=dict(size=sizes, color=bus["Area"], colorscale="Viridis",
                line=dict(width=1, color="white"), showscale=True,
                colorbar=dict(title="Area", thickness=12)),
    text=bus_text, hoverinfo="text",
    customdata=bus["Bus ID"].values,
    showlegend=False, name="Bus (click)")

In [ ]:
fig = go.FigureWidget(data=[edge_trace, line_trace, bus_trace])
fig.update_layout(
    title="RTS-GMLC Network - click a bus, or a line's square midpoint",
    xaxis=dict(title="Longitude", showgrid=False, zeroline=False),
    yaxis=dict(title="Latitude", showgrid=False, zeroline=False,
               scaleanchor="x", scaleratio=1.0),
    hovermode="closest", height=720,
    plot_bgcolor="white", margin=dict(l=40, r=40, t=60, b=40),
)

out = widgets.Output(layout=widgets.Layout(border="1px solid #ddd", padding="8px",
                                           max_height="420px", overflow="auto"))


def on_bus_click(trace, points, state):
    if not points.point_inds:
        return
    bid = int(trace.customdata[points.point_inds[0]])
    row = bus.loc[bus["Bus ID"] == bid].iloc[0]
    out.clear_output()
    with out:
        print(f"BUS {bid} - {row['Bus Name']}  |  {row['BaseKV']} kV  |  "
              f"Area {row['Area']}  |  Load {row['MW Load']} MW")
        g = gen_by_bus.get(bid)
        if g is None or len(g) == 0:
            print("No generators at this bus.")
        else:
            print(f"{len(g)} generator(s), total PMax = {g['PMax MW'].sum():.1f} MW")
            display(g)


def on_line_click(trace, points, state):
    if not points.point_inds:
        return
    r = branch_plot.iloc[int(trace.customdata[points.point_inds[0]])]
    out.clear_output()
    with out:
        print(f"LINE {r['UID']}")
        print(f"  From Bus   : {int(r['From Bus'])}")
        print(f"  To Bus     : {int(r['To Bus'])}")
        print(f"  LTE Rating : {r['LTE Rating']} MVA")
        print(f"  STE Rating : {r['STE Rating']} MVA")
        print(f"  (Cont Rating: {r['Cont Rating']} MVA, Length: {r['Length']})")


fig.data[2].on_click(on_bus_click)
fig.data[1].on_click(on_line_click)

widgets.VBox([fig, out])

    'data': [{'hoverinfo': 'skip',
              'line': {'color': 'rgba(120,120…

## Static fallback

If `FigureWidget` click events are unavailable in your Jupyter frontend, use this plain figure —
hovering a bus or a line midpoint shows the same fields.

In [ ]:
static = go.Figure(data=[edge_trace, line_trace, bus_trace])
static.update_layout(
    title="RTS-GMLC Network (hover for detail)",
    xaxis=dict(title="Longitude", showgrid=False, zeroline=False),
    yaxis=dict(title="Latitude", showgrid=False, zeroline=False,
               scaleanchor="x", scaleratio=1.0),
    hovermode="closest", height=720, plot_bgcolor="white")
static.show()

## Direct lookups

In [ ]:
def generators_at(bus_id):
    return gen_by_bus.get(int(bus_id), pd.DataFrame(columns=GEN_COLS))


def line_info(uid):
    return branch.loc[branch["UID"] == uid,
                      ["UID", "From Bus", "To Bus", "Cont Rating", "LTE Rating", "STE Rating"]]


display(generators_at(101))
display(line_info("A1"))